<a href="https://colab.research.google.com/github/iles-mcgill/EXSU500-Paediatric-HGGs-AI/blob/nnunet/04_nnunet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q nnunetv2
import os, json, re, requests, shutil, torch

DATA_DIR = '/content/drive/MyDrive/BraTS-PEDs-full/Training'
os.environ['nnUNet_raw']          = '/content/nnunet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnunet_preprocessed'
os.environ['nnUNet_results']      = '/content/drive/MyDrive/nnunet_results'
os.environ['nnUNet_n_proc_DA']    = '2'
os.environ['nnUNet_compile']      = 'false'
for k in ['nnUNet_raw', 'nnUNet_preprocessed', 'nnUNet_results']:
    os.makedirs(os.environ[k], exist_ok=True)

# your team's split
api = ('https://api.github.com/repos/iles-mcgill/EXSU500-Paediatric-HGGs-AI/'
       'contents/splits?ref=ansartlemussov/dataset-attribution-safeguards')
split = {}
for item in requests.get(api).json():
    if item['type'] == 'file':
        ids = sorted(set(re.findall(r'BraTS-PED-\d{5}-\d{3}',
                                    requests.get(item['download_url']).text)))
        for part in ['train', 'val', 'test']:
            if part in item['name'].lower() and ids:
                split[part] = ids
print('Split:', {k: len(v) for k, v in split.items()})   # 179 / 39 / 39

# dataset folder (train + val only)
DS = f"{os.environ['nnUNet_raw']}/Dataset501_BraTSPEDs"
os.makedirs(f'{DS}/imagesTr', exist_ok=True); os.makedirs(f'{DS}/labelsTr', exist_ok=True)
for c in split['train'] + split['val']:
    for i, ch in enumerate(['t1n', 't1c', 't2w', 't2f']):
        dst = f'{DS}/imagesTr/{c}_{i:04d}.nii.gz'
        if not os.path.exists(dst): os.symlink(f'{DATA_DIR}/{c}/{c}-{ch}.nii.gz', dst)
    dst = f'{DS}/labelsTr/{c}.nii.gz'
    if not os.path.exists(dst): os.symlink(f'{DATA_DIR}/{c}/{c}-seg.nii.gz', dst)
json.dump({'channel_names': {'0': 'T1', '1': 'T1c', '2': 'T2', '3': 'FLAIR'},
           'labels': {'background': 0, 'ET': 1, 'NET': 2, 'CC': 3, 'ED': 4},
           'numTraining': 218, 'file_ending': '.nii.gz'},
          open(f'{DS}/dataset.json', 'w'), indent=2)
print('images:', len(os.listdir(f'{DS}/imagesTr')), '(872)  labels:',
      len(os.listdir(f'{DS}/labelsTr')), '(218)')

Mounted at /content/drive
Split: {'test': 39, 'train': 179, 'val': 39}
images: 872 (872)  labels: 218 (218)


In [ ]:
!nnUNetv2_plan_and_preprocess -d 501 -c 3d_fullres -np 2
json.dump([{'train': split['train'], 'val': split['val']}],
          open(f"{os.environ['nnUNet_preprocessed']}/Dataset501_BraTSPEDs/splits_final.json", 'w'))
print('Split saved')

Fingerprint extraction...
Dataset501_BraTSPEDs
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
Extracting dataset fingerprint: 100% 218/218 [10:08<00:00,  2.79s/it]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [155.  240.  223.5], 3d_lowres: [155, 240, 224]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 56, 'patch_size': (np.int64(256), np.int64(224)), 'median_image_size_in_voxels': array([240. , 223.5]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization

In [ ]:
FOLD = (f"{os.environ['nnUNet_results']}/Dataset501_BraTSPEDs/"
        "nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres/fold_0")

def epoch_of(name):
    p = f'{FOLD}/{name}'
    return torch.load(p, map_location='cpu', weights_only=False)['current_epoch'] if os.path.exists(p) else None

if os.path.exists(f'{FOLD}/checkpoint_final.pth'):
    print('Training already FINISHED, nothing to do. Tell Claude.')
else:
    best, latest = epoch_of('checkpoint_best.pth'), epoch_of('checkpoint_latest.pth')
    print('best save:', best, '| latest save:', latest)
    if best is not None and (latest is None or best > latest):
        shutil.copy(f'{FOLD}/checkpoint_best.pth', f'{FOLD}/checkpoint_latest.pth')
    print('Resuming from epoch', max(e for e in [best, latest, 0] if e is not None))
    !nnUNetv2_train 501 3d_fullres 0 -tr nnUNetTrainer_100epochs --c

best save: 84 | latest save: 84
Resuming from epoch 84

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(
Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-11-17 17:29:32.648223: do_dummy_2d_data_aug: False
2026-11-17 17

In [3]:
# TEST SET: nnU-Net predicts the 39 test patients. Run ONCE (models are frozen).
TEST_IN  = '/content/nnunet_test_input'
TEST_OUT = '/content/drive/MyDrive/EXSU500_results/nnunet_test_pred'
os.makedirs(TEST_IN, exist_ok=True)
os.makedirs(TEST_OUT, exist_ok=True)

for c in split['test']:
    for i, ch in enumerate(['t1n', 't1c', 't2w', 't2f']):
        dst = f'{TEST_IN}/{c}_{i:04d}.nii.gz'
        if not os.path.exists(dst):
            os.symlink(f'{DATA_DIR}/{c}/{c}-{ch}.nii.gz', dst)
print(len(os.listdir(TEST_IN)), 'test images (should be 156)')

!nnUNetv2_predict -i {TEST_IN} -o {TEST_OUT} -d 501 -c 3d_fullres -f 0 \
    -tr nnUNetTrainer_100epochs -chk checkpoint_final.pth -npp 2 -nps 2

n = len([f for f in os.listdir(TEST_OUT) if f.endswith('.nii.gz')])
print('Predictions saved:', n, '(should be 39)')

156 test images (should be 156)

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(
There are 39 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 39 cases that I would like to predict

Predicting BraTS-PED-00003-000:
perform_everything_on_device: True
100% 12/12 [00:14<00:00,  1.22s/it]
sending off prediction to background worker for resampling and export
done with BraTS-PED-00003-000

Predicting BraTS-PED-00018-